# Cinéfilo: a base do MP3 como grafo de propriedades no Memgraph

**Mini-Projeto 4, Sistemas Baseados em Conhecimento (UFPB).**

No Mini-Projeto 3, o Cinéfilo virou uma ontologia em RDF, RDFS e OWL, consultada com rdflib e SPARQL.
Aqui, **a mesma base** é remodelada como **grafo de propriedades** (nós, rótulos, relacionamentos e
propriedades), guardada no **Memgraph** e consultada com **Cypher**.

A mudança mais importante: o Memgraph **não tem reasoner**. Tudo o que no MP3 era deduzido pelo OWL
(tipos herdados, inversas, transitividade, simetria) aqui é resolvido **pela própria consulta**,
percorrendo o grafo. É o que a aula chama de *"LPG: inferência fraca; as regras de inferência ficam na
aplicação"*.

| Seção | Conteúdo |
|---|---|
| 1 | Conexão com o Memgraph e funções auxiliares |
| 2 | A base: conferência da carga, modelo e taxonomia |
| 3 | **Parte 1:** as consultas `g.triples()` do MP3 em Cypher (T1 a T7) |
| 4 | O que era inferência vira consulta (I1 a I6) e verificações de consistência (V1, V2) |
| 5 | **Parte 2:** as consultas SPARQL do MP3 em Cypher (S1 a S13) |
| 6 | Resumo das consultas |

**Antes de executar:** o Memgraph precisa estar rodando (`docker compose up -d`) e a base carregada
(`python src/carregar.py`). Se a base estiver vazia, a seção 2 faz a carga. Depois, use **Run All**.

## 1. Conexão com o Memgraph

O Memgraph fala o protocolo **Bolt** (porta 7687), então o Python usa o driver oficial `neo4j`, como na
aula. O Memgraph não exige usuário e senha por padrão, daí o par vazio em `AUTH`.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("neo4j") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neo4j>=5.8,<6"])

In [ ]:
import json

from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("", "")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Conectado ao Memgraph em", URI)

Funções auxiliares:

- `executar(cypher, tx=None)` roda uma consulta e devolve as colunas e as linhas. Com `tx`, roda dentro
  de uma transação (usado na seção 5.4).
- `mostrar(cypher, tx=None)` roda e imprime o resultado como tabela.
- `valor(cypher)` devolve o primeiro valor do resultado (útil para contagens e respostas sim/não).

In [ ]:
def executar(cypher, tx=None, **parametros):
    """Roda uma consulta Cypher e devolve (colunas, linhas)."""
    if tx is None:
        registros, _, colunas = driver.execute_query(cypher, parameters_=parametros, database_="memgraph")
        return list(colunas), [list(r.values()) for r in registros]
    resultado = tx.run(cypher, parametros)
    linhas = [list(r.values()) for r in resultado]
    return list(resultado.keys()), linhas


def formatar(v):
    if v is None:
        return "—"
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    if isinstance(v, list):
        return " + ".join(formatar(x) for x in v) if v else "—"
    if isinstance(v, dict):
        return ", ".join(f"{k}={formatar(x)}" for k, x in v.items())
    return str(v)


def mostrar(cypher, tx=None, **parametros):
    """Roda a consulta e imprime o resultado como tabela de texto."""
    colunas, linhas = executar(cypher, tx, **parametros)
    texto = [[formatar(v) for v in linha] for linha in linhas]
    larguras = [max([len(c)] + [len(l[i]) for l in texto]) for i, c in enumerate(colunas)]
    print("  ".join(c.ljust(w) for c, w in zip(colunas, larguras)))
    print("  ".join("-" * w for w in larguras))
    for linha in texto:
        print("  ".join(v.ljust(w) for v, w in zip(linha, larguras)))
    print(f"({len(linhas)} linha(s))")
    return linhas


def valor(cypher, tx=None, **parametros):
    """Primeiro valor da primeira linha do resultado."""
    return executar(cypher, tx, **parametros)[1][0][0]


def cabecalho(codigo, proposito):
    print(f"[Consulta {codigo}] {proposito}")
    print("-" * 78)

## 2. A base

### 2.1 Carga e conferência

A base é criada pelo script `src/carregar.py`, que apaga o banco e executa `cypher/01_carga.cypher`. A
célula abaixo só carrega se o banco estiver vazio. Assim, se a base já existia antes de um
`docker compose restart memgraph`, ela é apenas conferida, o que demonstra a **persistência**.

A conferência compara cada rótulo e cada tipo de relacionamento com o conteúdo do `cinema.ttl` do MP3:
17 filmes, 36 pessoas, 8 estúdios e 7 prêmios (as 68 instâncias do MP3), mais os 7 gêneros.

In [ ]:
sys.path.insert(0, "src")
from carregar import carregar, conferir

if valor("MATCH (n) RETURN count(n)") == 0:
    print("Banco vazio: executando a carga.\n")
    carregar(driver)
else:
    print("Base já existente no Memgraph: só conferindo.\n")
    conferir(driver)

### 2.2 O modelo

Cada nó tem um ou mais **rótulos** e um conjunto de **propriedades**. Os rótulos empilhados guardam a
hierarquia do MP3: um diretor é `Pessoa:Profissional:Diretor`, como `Diretor ⊂ Profissional ⊂ Pessoa`.

In [ ]:
cabecalho("M1", "Combinações de rótulos e quantos nós há em cada uma")
mostrar("""
    MATCH (n)
    RETURN labels(n) AS rotulos, count(*) AS nos
    ORDER BY nos DESC
""")

Os relacionamentos ligam sempre os mesmos tipos de nó. É o equivalente ao `rdfs:domain` e ao
`rdfs:range` do MP3: a consulta abaixo mostra, para cada relacionamento, de onde ele sai e aonde chega.

In [ ]:
cabecalho("M2", "Esquema: cada relacionamento, de onde sai e aonde chega")
mostrar("""
    MATCH (a)-[r]->(b)
    WITH type(r) AS relacionamento,
         [l IN labels(a) WHERE l IN ['Filme', 'Pessoa', 'Estudio', 'Premio', 'Genero']][0] AS de,
         [l IN labels(b) WHERE l IN ['Filme', 'Pessoa', 'Estudio', 'Premio', 'Genero']][0] AS para
    RETURN relacionamento, de, para, count(*) AS quantidade
    ORDER BY relacionamento
""")

### 2.3 A taxonomia de gêneros

No MP3, os gêneros eram **classes** (`Drama rdfs:subClassOf Ficcao`). Aqui eles são **nós**, ligados
por `SUBGENERO_DE`. A hierarquia continua no grafo, e as consultas a percorrem. A disjunção entre
Ficção e Documentário também é um relacionamento, `DISJUNTO_DE`, usado pela verificação V1.

O ano em que cada prêmio foi entregue fica **na aresta** `GANHOU`: é o recurso que a aula aponta como o
grande diferencial do grafo de propriedades, porque em RDF exigiria reificação.

In [ ]:
cabecalho("M3", "Hierarquia de gêneros e quantos filmes cada um tem diretamente")
mostrar("""
    MATCH (g:Genero)
    OPTIONAL MATCH (g)-[:SUBGENERO_DE]->(pai:Genero)
    OPTIONAL MATCH (f:Filme)-[:DO_GENERO]->(g)
    RETURN g.nome AS genero, pai.nome AS subgenero_de, count(f) AS filmes
    ORDER BY subgenero_de, genero
""")

## 3. Parte 1: as consultas `g.triples()` do MP3 em Cypher

No MP3, `g.triples((s, p, o))` casava **um padrão de tripla**, com `None` como coringa. Em Cypher, o
equivalente é um `MATCH` com um só relacionamento, em que as partes fixas têm `id` e as livres são
variáveis. Há uma diferença de modelo: no grafo de propriedades, valores como nota e ano são
**propriedades do nó**, e não triplas. Por isso a T3 consulta uma propriedade, e a T4 devolve rótulos e
propriedades separados dos relacionamentos.

As consultas desta seção usam só o que está gravado, sem percorrer hierarquias, como no MP3.

| Consulta | Padrão do MP3 | Equivalente em Cypher |
|---|---|---|
| T1 | `(s, p, None)` | `(nó fixo)-[:REL]->(livre)` |
| T2 | `(None, p, o)` | `(livre)-[:REL]->(nó fixo)` |
| T3 | `(None, p, None)` | a propriedade `nota` de todos os filmes |
| T4 | `(s, None, None)` | rótulos, propriedades e arestas de saída de um nó |
| T5 | `(None, None, o)` | `(livre)-[qualquer]->(nó fixo)` |
| T6 | `(s, None, o)` | `(nó fixo)-[qualquer]->(nó fixo)` |
| T7 | `(None, rdf:type, o)` | filmes ligados **diretamente** ao gênero Ficção |

In [ ]:
# Busca:    os atores ligados a Interestelar por TEM_NO_ELENCO.
# Esperado: Anne Hathaway, Jessica Chastain e Matthew McConaughey.
cabecalho("T1", "Elenco de Interestelar  (nó fixo, relacionamento fixo, destino livre)")
mostrar("""
MATCH (:Filme {id: 'interestelar'})-[:TEM_NO_ELENCO]->(a:Pessoa)
RETURN a.nome AS ator
ORDER BY ator
""")

In [ ]:
# Busca:    os filmes com DIRIGIDO_POR apontando para Villeneuve.
# Esperado: A Chegada, Blade Runner 2049, Duna e Duna: Parte Dois.
cabecalho("T2", "Filmes dirigidos por Denis Villeneuve  (origem livre, relacionamento e destino fixos)")
mostrar("""
MATCH (f:Filme)-[:DIRIGIDO_POR]->(:Pessoa {id: 'villeneuve'})
RETURN f.titulo AS filme
ORDER BY filme
""")

In [ ]:
# Busca:    a propriedade nota de todos os filmes, ordenada.
# Esperado: Batman: O Cavaleiro das Trevas (9.0), A Origem (8.8), Interestelar (8.7),
#           Duna: Parte Dois (8.5) e Parasita (8.5).
cabecalho("T3", "Cinco maiores notas no IMDb  (propriedade fixa, nós livres)")
mostrar("""
MATCH (f:Filme)
WHERE f.nota IS NOT NULL
RETURN f.titulo AS filme, f.nota AS nota
ORDER BY nota DESC, filme
LIMIT 5
""")

In [ ]:
# Busca:    tudo o que sai do nó de Eastwood: rótulos, propriedades e arestas de saída.
# Esperado: rótulos Pessoa, Profissional e Diretor; nascimento 1930; nacionalidade EUA;
#           COLABOROU_COM -> Morgan Freeman. Nada diz que ele é ator: isso está numa
#           aresta que CHEGA nele (T5).
cabecalho("T4", "Tudo o que sai do nó de Clint Eastwood  (nó fixo, o resto livre)")
mostrar("""
MATCH (p:Pessoa {id: 'eastwood'})
OPTIONAL MATCH (p)-[r]->(x)
RETURN labels(p) AS rotulos, properties(p) AS propriedades,
       collect(type(r) + ' -> ' + x.nome) AS relacionamentos_de_saida
""")

In [ ]:
# Busca:    todas as arestas que chegam ao nó de Eastwood.
# Esperado: Menina de Ouro, duas vezes: DIRIGIDO_POR e TEM_NO_ELENCO.
cabecalho("T5", "Onde Clint Eastwood aparece como destino  (destino fixo, o resto livre)")
mostrar("""
MATCH (x)-[r]->(:Pessoa {id: 'eastwood'})
RETURN x.titulo AS origem, type(r) AS relacionamento
ORDER BY relacionamento
""")

In [ ]:
# Busca:    qualquer relacionamento direto de Nolan para Villeneuve.
# Esperado: um só, INFLUENCIOU.
cabecalho("T6", "Que relacionamento liga Nolan a Villeneuve  (origem e destino fixos)")
mostrar("""
MATCH (:Pessoa {id: 'nolan'})-[r]->(:Pessoa {id: 'villeneuve'})
RETURN type(r) AS relacionamento
""")

In [ ]:
# Busca:    filmes com DO_GENERO apontando direto para Ficção.
# Esperado: 0. Cada filme aponta para o gênero mais específico (Drama, Terror...).
#           Que um Drama é Ficção está na hierarquia SUBGENERO_DE, que esta consulta
#           não percorre. A seção 4 resolve isso.
cabecalho("T7", "Filmes ligados diretamente ao gênero Ficção  (sem percorrer a hierarquia)")
mostrar("""
MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'Ficcao'})
RETURN count(f) AS filmes
""")

## 4. O que era inferência vira consulta

No MP3, o reasoner OWL-RL acrescentou 608 triplas ao grafo antes das consultas. O Memgraph não tem
reasoner, e nada é acrescentado. Cada regra do MP3 é aplicada **na hora da consulta**, percorrendo o
grafo. A célula abaixo faz cada pergunta duas vezes:

- **direto:** só a aresta gravada, como o "antes" do MP3;
- **com a regra:** a consulta aplica a regra do MP3, como o "depois".

In [ ]:
# Cada item: (construção do MP3, pergunta, consulta direta, consulta com a regra)
COMPARACOES = {
    "I1": (
        "rdfs:subClassOf  ->  caminho SUBGENERO_DE*",
        "que filmes são de Ficção?",
        """
MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'Ficcao'})
RETURN f.titulo AS nome""",
        """
MATCH (f:Filme)-[:DO_GENERO]->(:Genero)-[:SUBGENERO_DE*]->(:Genero {id: 'Ficcao'})
RETURN DISTINCT f.titulo AS nome""",
    ),
    "I2": (
        "rdfs:domain / rdfs:range  ->  papel pelo relacionamento",
        "quais são os papéis de Christopher Nolan?",
        """
MATCH (p:Pessoa {id: 'nolan'})
UNWIND labels(p) AS nome
RETURN nome""",
        """
MATCH (p:Pessoa {id: 'nolan'})
UNWIND labels(p) AS nome
RETURN nome
UNION
MATCH (:Pessoa {id: 'nolan'})<-[r]-(:Filme)
RETURN CASE type(r)
         WHEN 'DIRIGIDO_POR' THEN 'Diretor'
         WHEN 'TEM_NO_ELENCO' THEN 'Ator'
         WHEN 'ROTEIRIZADO_POR' THEN 'Roteirista'
         WHEN 'TRILHA_SONORA_DE' THEN 'Compositor'
       END AS nome""",
    ),
    "I3": (
        "owl:inverseOf  ->  a mesma aresta, percorrida ao contrário",
        "o que Christopher Nolan dirigiu?",
        """
MATCH (f:Filme)-[:DIRIGIDO_POR]->(:Pessoa {id: 'nolan'})
RETURN f.titulo AS nome""",
        """
MATCH (:Pessoa {id: 'nolan'})<-[:DIRIGIDO_POR]-(f:Filme)
RETURN f.titulo AS nome""",
    ),
    "I4": (
        "owl:TransitiveProperty  ->  caminho INFLUENCIOU*",
        "quem Stanley Kubrick influenciou?",
        """
MATCH (:Pessoa {id: 'kubrick'})-[:INFLUENCIOU]->(x:Pessoa)
RETURN x.nome AS nome""",
        """
MATCH (:Pessoa {id: 'kubrick'})-[:INFLUENCIOU*]->(x:Pessoa)
RETURN DISTINCT x.nome AS nome""",
    ),
    "I5": (
        "owl:SymmetricProperty  ->  relacionamento sem direção",
        "com quem Hans Zimmer colaborou?",
        """
MATCH (:Pessoa {id: 'zimmer'})-[:COLABOROU_COM]->(x:Pessoa)
RETURN x.nome AS nome""",
        """
MATCH (:Pessoa {id: 'zimmer'})-[:COLABOROU_COM]-(x:Pessoa)
RETURN x.nome AS nome""",
    ),
    "I6": (
        "rdfs:subPropertyOf + transitiva  ->  caminho SEQUENCIA_DE*",
        "Toy Story 3 vem depois de quais filmes?",
        """
MATCH (:Filme {id: 'toyStory3'})-[:SEQUENCIA_DE]->(x:Filme)
RETURN x.titulo AS nome""",
        """
MATCH (:Filme {id: 'toyStory3'})-[:SEQUENCIA_DE*]->(x:Filme)
RETURN DISTINCT x.titulo AS nome""",
    ),
}


def comparar(codigo):
    construcao, pergunta, direto, com_regra = COMPARACOES[codigo]
    antes = sorted(linha[0] for linha in executar(direto)[1])
    depois = sorted(set(linha[0] for linha in executar(com_regra)[1]))
    novos = [x for x in depois if x not in antes]
    print(f"[{codigo}] {construcao}: {pergunta}")
    print(f"  direto        ({len(antes):>2}): {', '.join(antes) or '—'}")
    print(f"  com a regra   ({len(depois):>2}): {', '.join(depois) or '—'}")
    print(f"  só pela regra: {', '.join(novos) or '—'}\n")


for codigo in COMPARACOES:
    comparar(codigo)

Leitura do resultado:

- **I1, subClassOf:** é a resposta da T7. Direto, 0 filmes; percorrendo `SUBGENERO_DE*`, 16. O `*`
  segue a hierarquia por quantos níveis houver. Só Free Solo fica de fora, porque é documentário.
- **I2, domain/range:** Nolan foi gravado com o rótulo `Diretor`. O papel de roteirista vem do
  relacionamento `ROTEIRIZADO_POR` que chega nele, o mesmo raciocínio do `rdfs:range`.
- **I3, inverseOf:** as duas consultas dão o mesmo resultado. No grafo de propriedades **não existe
  inversa**: a mesma aresta `DIRIGIDO_POR` é percorrida nos dois sentidos. Nada é duplicado.
- **I4, transitiva:** `INFLUENCIOU*` fecha a cadeia Kubrick → Nolan → Villeneuve.
- **I5, simétrica:** a colaboração foi gravada a partir de Nolan e de Villeneuve. Sem a seta, ela vale
  nos dois sentidos.
- **I6, franquia:** a sequência direta não é transitiva, mas "vir depois na franquia" é. O caminho
  `SEQUENCIA_DE*` responde a segunda pergunta sem mudar o dado.
- **equivalentClass (schema.org):** é tratada na consulta S10, que exporta o catálogo nesse vocabulário.
- **FunctionalProperty:** não precisa de regra. Uma propriedade de nó guarda **um único valor**: um filme
  não consegue ter duas notas. O modelo já garante a regra.

### 4.1 Verificações de consistência

`owl:disjointWith` e `owl:AsymmetricProperty` não geravam fatos: elas proibiam combinações, e o reasoner
acusava as violações. Aqui, cada uma vira uma **consulta que procura violações**. Na base correta,
as duas devolvem 0 linhas.

A V1 é genérica: ela lê no próprio grafo quais gêneros são disjuntos (`DISJUNTO_DE`) e procura filmes
que caiam nos dois, subindo a hierarquia de gêneros.

In [ ]:
# Esperado na base correta: 0 linhas.
CONSULTA_V1 = """
MATCH (f:Filme)-[:DO_GENERO]->(g:Genero)
OPTIONAL MATCH (g)-[:SUBGENERO_DE*]->(acima:Genero)
WITH f, collect(DISTINCT g) + collect(DISTINCT acima) AS categorias
MATCH (a:Genero)-[:DISJUNTO_DE]->(b:Genero)
WHERE a IN categorias AND b IN categorias
RETURN f.titulo AS filme, a.nome AS categoria_1, b.nome AS categoria_2
"""
cabecalho("V1", "Verificação de owl:disjointWith: filmes em dois gêneros disjuntos")
mostrar(CONSULTA_V1)

In [ ]:
# Esperado na base correta: 0 linhas. O filtro a.id < b.id evita listar o mesmo par duas vezes,
# uma a partir de cada filme.
CONSULTA_V2 = """
MATCH (a:Filme)-[:SEQUENCIA_DE]->(b:Filme)-[:SEQUENCIA_DE]->(a)
WHERE a.id < b.id
RETURN a.titulo AS filme_1, b.titulo AS filme_2
"""
cabecalho("V2", "Verificação de owl:AsymmetricProperty: sequências em ciclo")
mostrar(CONSULTA_V2)

Para mostrar que as verificações funcionam, a célula abaixo insere os **mesmos dois erros do MP3**
dentro de uma transação, roda as verificações e **desfaz tudo** com `rollback`. A base não é alterada.

1. Free Solo, um documentário, marcado também como Drama.
2. Toy Story como sequência de Toy Story 2, que já é sequência de Toy Story.

In [ ]:
with driver.session(database="memgraph") as sessao:
    tx = sessao.begin_transaction()
    try:
        tx.run("""MATCH (f:Filme {id: 'freeSolo'}), (g:Genero {id: 'Drama'})
                  CREATE (f)-[:DO_GENERO]->(g)""").consume()
        tx.run("""MATCH (a:Filme {id: 'toyStory'}), (b:Filme {id: 'toyStory2'})
                  CREATE (a)-[:SEQUENCIA_DE]->(b)""").consume()
        print("Com os dois erros inseridos (dentro da transação):\n")
        cabecalho("V1", "filmes em gêneros disjuntos")
        mostrar(CONSULTA_V1, tx)
        print()
        cabecalho("V2", "sequências em ciclo")
        mostrar(CONSULTA_V2, tx)
    finally:
        tx.rollback()

print("\nDepois do rollback, a base voltou ao normal:",
      "V1 =", len(executar(CONSULTA_V1)[1]), "linha(s), V2 =", len(executar(CONSULTA_V2)[1]), "linha(s)")

## 5. Parte 2: as consultas SPARQL do MP3 em Cypher

O Cypher não tem formas separadas como o SPARQL (SELECT, ASK, CONSTRUCT). Tudo é `MATCH ... RETURN`, e
cada forma do MP3 ganha um equivalente:

| MP3 (SPARQL) | MP4 (Cypher) | Consultas |
|---|---|---|
| SELECT com FILTER | `MATCH ... WHERE ... RETURN` | S1 |
| SELECT com ORDER BY / LIMIT | `ORDER BY ... LIMIT` | S2 |
| agregação, GROUP BY, HAVING | `count`, `avg`... (o agrupamento é implícito); o HAVING vira `WITH ... WHERE` | S3, S4 |
| VALUES + GROUP_CONCAT | `CASE` + `collect` | S5 |
| OPTIONAL | `OPTIONAL MATCH` | S6 |
| caminho `influenciou+` | caminho variável `[:INFLUENCIOU*]` | S7 |
| ASK | `RETURN count(*) > 0` | S8, S9 |
| CONSTRUCT | `RETURN` de um mapa em formato schema.org (JSON-LD) | S10 |
| INSERT DATA | `CREATE` | S11 |
| DELETE/INSERT | `SET` | S12 |
| DELETE | `DETACH DELETE` | S13 |

**Travessias** (dois ou mais relacionamentos encadeados): S1, S4, S7, S8b e S9, além de I1, I4, I6 e V1
na seção 4.

### 5.1 Consultas de leitura

In [ ]:
# Ficção de qualquer subgênero: o caminho DO_GENERO + SUBGENERO_DE* sobe a hierarquia
# (no MP3 isso dependia do reasoner). WITH DISTINCT evita repetir filmes de dois gêneros.
# Esperado: 6 filmes, com A Origem (8.8) no topo.
cabecalho("S1", "Ficção lançada a partir de 2010 com nota ≥ 8.3  (WHERE; travessia)")
mostrar("""
MATCH (f:Filme)-[:DO_GENERO]->(:Genero)-[:SUBGENERO_DE*]->(:Genero {id: 'Ficcao'})
WITH DISTINCT f
WHERE f.ano >= 2010 AND f.nota >= 8.3
MATCH (f)-[:DIRIGIDO_POR]->(d:Pessoa)
RETURN f.titulo AS filme, f.ano AS ano, f.nota AS nota, d.nome AS diretor
ORDER BY nota DESC, ano
""")

In [ ]:
# Esperado: Oppenheimer (180 min) no topo, seguido de Interestelar (169).
cabecalho("S2", "Os 5 filmes mais longos  (ORDER BY + LIMIT)")
mostrar("""
MATCH (f:Filme)
RETURN f.titulo AS filme, f.duracao AS duracao, f.ano AS ano
ORDER BY duracao DESC
LIMIT 5
""")

In [ ]:
# Agrupamento implícito por d; o HAVING do SPARQL vira WITH ... WHERE.
# No MP3 a consulta usava a inversa 'dirigiu'; aqui a mesma aresta DIRIGIDO_POR basta.
# A média é arredondada para 2 casas, para empates como 8.10 serem exatos.
# Esperado: Nolan e Villeneuve com 4 filmes cada; Nolan com a maior média (8.70).
cabecalho("S3", "Diretores com 2+ filmes: quantidade, nota média e período  (agregação)")
mostrar("""
MATCH (f:Filme)-[:DIRIGIDO_POR]->(d:Pessoa)
WITH d, count(f) AS filmes, round(avg(f.nota) * 100) / 100.0 AS media,
     min(f.ano) AS primeiro, max(f.ano) AS ultimo
WHERE filmes >= 2
RETURN d.nome AS diretor, filmes, media, primeiro, ultimo
ORDER BY media DESC, filmes DESC, diretor
""")

In [ ]:
# Os gêneros não estão escritos na consulta: ela pergunta ao grafo quais nós são
# subgêneros de Ficção. Um filme pode contar em dois gêneros (Interestelar).
# Esperado: Ficção científica com 7 filmes, Drama com 6.
cabecalho("S4", "Filmes e nota média por gênero de ficção  (agregação; travessia)")
mostrar("""
MATCH (f:Filme)-[:DO_GENERO]->(g:Genero)-[:SUBGENERO_DE]->(:Genero {id: 'Ficcao'})
RETURN g.nome AS genero, count(f) AS filmes, round(avg(f.nota) * 100) / 100.0 AS media
ORDER BY filmes DESC, media DESC
""")

In [ ]:
# O papel de cada pessoa vem do relacionamento que chega nela (o rdfs:range do MP3).
# A coluna 'declarado' mostra o rótulo gravado; 'papeis', os papéis pelas arestas.
# Esperado: Bong Joon-ho, Christopher Nolan, Jordan Peele e Stanley Kubrick como
# diretores-roteiristas, e Clint Eastwood como ator e diretor.
cabecalho("S5", "Profissionais que acumulam mais de um papel  (CASE + collect)")
mostrar("""
MATCH (p:Pessoa)<-[r]-(:Filme)
WITH DISTINCT p, CASE type(r)
                   WHEN 'DIRIGIDO_POR' THEN 'Diretor'
                   WHEN 'TEM_NO_ELENCO' THEN 'Ator'
                   WHEN 'ROTEIRIZADO_POR' THEN 'Roteirista'
                   WHEN 'TRILHA_SONORA_DE' THEN 'Compositor'
                 END AS papel
WITH p, papel ORDER BY papel
WITH p, collect(papel) AS papeis
WHERE size(papeis) > 1
RETURN p.nome AS pessoa,
       [l IN labels(p) WHERE l IN ['Diretor', 'Ator', 'Roteirista', 'Compositor']] AS declarado,
       papeis
ORDER BY pessoa
""")

In [ ]:
# OPTIONAL MATCH mantém A Chegada, que não tem prêmio registrado.
# A coluna 'cerimonia' é a propriedade 'ano' da ARESTA GANHOU.
# Esperado: 8 linhas (Duna aparece duas vezes); A Chegada com '—'.
cabecalho("S6", "Ficção científica e prêmios recebidos, quando houver  (OPTIONAL MATCH)")
mostrar("""
MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'FiccaoCientifica'})
OPTIONAL MATCH (f)-[g:GANHOU]->(p:Premio)
RETURN f.titulo AS filme, f.ano AS ano, p.nome AS premio, g.ano AS cerimonia
ORDER BY ano, premio
""")

In [ ]:
# O * segue INFLUENCIOU por quantos saltos houver; size(r) conta os saltos.
# Esperado: Villeneuve (2 saltos, via Nolan), Nolan e Peele (1 salto).
cabecalho("S7", "Diretores influenciados por Kubrick, direta ou indiretamente  (caminho variável; travessia)")
mostrar("""
MATCH (:Pessoa {id: 'kubrick'})-[r:INFLUENCIOU*]->(x:Pessoa)
RETURN x.nome AS influenciado, x.nascimento AS nascimento, size(r) AS saltos
ORDER BY nascimento
""")

### 5.2 Perguntas de sim ou não (o ASK do MP3)

O Cypher não tem ASK. O equivalente é contar as soluções do padrão e perguntar se a contagem é maior que
zero: `RETURN count(*) > 0`.

A S8 repete a comparação do MP3. Lá, a mesma pergunta dava `False` na base e `True` depois do reasoner.
Aqui não há reasoner: a diferença está **na consulta**, entre seguir um salto (`INFLUENCIOU`) e seguir
um caminho (`INFLUENCIOU*`).

In [ ]:
# Esperado: False com um salto (não há aresta Kubrick -> Villeneuve) e True com o caminho.
cabecalho("S8", "Stanley Kubrick influenciou Denis Villeneuve?")
print("  um salto (INFLUENCIOU):  ", valor("""
MATCH (:Pessoa {id: 'kubrick'})-[:INFLUENCIOU]->(:Pessoa {id: 'villeneuve'})
RETURN count(*) > 0 AS resposta
"""))
print("  caminho  (INFLUENCIOU*): ", valor("""
MATCH (:Pessoa {id: 'kubrick'})-[:INFLUENCIOU*]->(:Pessoa {id: 'villeneuve'})
RETURN count(*) > 0 AS resposta
"""))

In [ ]:
# O mesmo p nas duas pontas obriga a ser a mesma pessoa: dirigiu E atuou no mesmo filme.
# No MP3 isso usava as duas inversas; aqui são as duas arestas que saem do filme.
# Esperado: True (Clint Eastwood, em Menina de Ouro).
cabecalho("S9", "Existe alguém que atuou em um filme que ele mesmo dirigiu?  (ASK; travessia)")
print(" ", valor("""
MATCH (p:Pessoa)<-[:DIRIGIDO_POR]-(:Filme)-[:TEM_NO_ELENCO]->(p)
RETURN count(*) > 0 AS resposta
"""))

### 5.3 Exportação para schema.org (o CONSTRUCT do MP3)

O Cypher não devolve grafos como o CONSTRUCT. O equivalente é devolver cada filme como um **mapa** no
formato JSON-LD do schema.org, o vocabulário que os buscadores leem. O resultado é o mesmo recorte do
MP3: filmes com título, ano, duração, diretores e elenco.

In [ ]:
# Esperado: 17 filmes, cada um como um objeto schema:Movie.
cabecalho("S10", "Exportação do catálogo no formato schema.org (JSON-LD)  (equivalente ao CONSTRUCT)")
_, linhas = executar("""
MATCH (f:Filme)
OPTIONAL MATCH (f)-[:DIRIGIDO_POR]->(d:Pessoa)
WITH f, collect(DISTINCT d) AS diretores
OPTIONAL MATCH (f)-[:TEM_NO_ELENCO]->(a:Pessoa)
WITH f, diretores, collect(DISTINCT a) AS atores
RETURN {
  `@context`: 'https://schema.org',
  `@type`: 'Movie',
  `@id`: f.id,
  name: f.titulo,
  datePublished: f.ano,
  duration: 'PT' + toString(f.duracao) + 'M',
  director: [d IN diretores | {`@type`: 'Person', name: d.nome}],
  actor: [a IN atores | {`@type`: 'Person', name: a.nome}]
} AS filme
""")
filmes = [linha[0] for linha in linhas]
print(f"Filmes exportados: {len(filmes)}")
print("Todos do tipo schema:Movie?", all(f["@type"] == "Movie" for f in filmes))
print("\nAmostra (Interestelar):\n")
interestelar = next(f for f in filmes if f["@id"] == "interestelar")
print(json.dumps(interestelar, ensure_ascii=False, indent=2))

### 5.4 Atualizações

No MP3, os updates rodavam numa cópia do grafo em memória. Aqui o banco é **persistente**: um DELETE
apagaria os filmes de terror de verdade. Por isso as três atualizações rodam **dentro de uma
transação**, aberta na célula abaixo e **desfeita com rollback** no fim da seção. O efeito de cada
uma aparece na saída, e a base continua intacta para as próximas execuções.

> Se alguma célula desta seção for executada fora de ordem, execute de novo a partir da célula abaixo.

In [ ]:
sessao = driver.session(database="memgraph")
tx = sessao.begin_transaction()
print("Transação aberta. As alterações S11 a S13 só existem dentro dela, até o rollback.")

In [ ]:
# CREATE é o INSERT DATA do MP3. Tenet aponta para nós que já existem (Nolan, Göransson,
# Warner e os gêneros), localizados antes pelo MATCH.
# Diferença para o MP3: lá, a inversa 'nolan dirigiu tenet' só aparecia depois de rodar o
# reasoner de novo. Aqui não existe inversa a criar: a mesma aresta é percorrida ao contrário.
# Esperado: Nolan passa de 4 para 5 filmes, já na mesma transação.
cabecalho("S11", "CREATE: cadastrar o filme Tenet (Nolan, 2020)")
FILMES_NOLAN = """
MATCH (:Pessoa {id: 'nolan'})<-[:DIRIGIDO_POR]-(f:Filme)
RETURN f.titulo ORDER BY f.titulo
"""
antes = [l[0] for l in executar(FILMES_NOLAN, tx)[1]]
print(f"  antes:  {len(antes)} filmes de Nolan — {', '.join(antes)}")

tx.run("""
MATCH (nolan:Pessoa {id: 'nolan'}), (goransson:Pessoa {id: 'goransson'}),
      (warner:Estudio {id: 'warner'}),
      (fc:Genero {id: 'FiccaoCientifica'}), (acao:Genero {id: 'Acao'})
CREATE (tenet:Obra:Filme {id: 'tenet', titulo: 'Tenet', ano: 2020, duracao: 150, nota: 7.3}),
       (washington:Pessoa:Profissional:Ator {id: 'washington', nome: 'John David Washington',
                                             nascimento: 1984, nacionalidade: 'EUA'}),
       (pattinson:Pessoa:Profissional:Ator {id: 'pattinson', nome: 'Robert Pattinson',
                                            nascimento: 1986, nacionalidade: 'Reino Unido'}),
       (tenet)-[:DO_GENERO]->(fc),
       (tenet)-[:DO_GENERO]->(acao),
       (tenet)-[:DIRIGIDO_POR]->(nolan),
       (tenet)-[:ROTEIRIZADO_POR]->(nolan),
       (tenet)-[:TEM_NO_ELENCO]->(washington),
       (tenet)-[:TEM_NO_ELENCO]->(pattinson),
       (tenet)-[:TRILHA_SONORA_DE]->(goransson),
       (tenet)-[:PRODUZIDO_POR]->(warner)
""").consume()

depois = [l[0] for l in executar(FILMES_NOLAN, tx)[1]]
print(f"  depois: {len(depois)} filmes de Nolan — {', '.join(depois)}")

In [ ]:
# SET é o DELETE/INSERT do MP3. No MP3 a nota antiga precisava ser removida porque
# notaIMDb era FunctionalProperty e um INSERT deixaria duas notas. Aqui a nota é uma
# propriedade do nó, que guarda um único valor: o SET substitui o valor antigo.
# Esperado: antes 7.3, depois 7.4.
cabecalho("S12", "SET: atualizar a nota de Tenet para 7.4  (equivalente ao DELETE/INSERT)")
NOTA_TENET = "MATCH (f:Filme {id: 'tenet'}) RETURN f.nota"
print("  antes: ", valor(NOTA_TENET, tx))

tx.run("""
MATCH (f:Filme {id: 'tenet'})
SET f.nota = 7.4
""").consume()

print("  depois:", valor(NOTA_TENET, tx))

In [ ]:
# DETACH DELETE apaga o nó e TODOS os relacionamentos dele, nos dois sentidos.
# No MP3 a consulta precisava apagar explicitamente as triplas de entrada e de saída
# (com UNION); aqui o DETACH faz isso. As pessoas (Kubrick, Peele, Nicholson...)
# continuam no banco; só as ligações com os filmes removidos somem.
# Esperado: 2 -> 0 filmes de terror; 12 -> 0 relacionamentos ligados a eles;
#           Kubrick passa de 2 filmes para 1.
cabecalho("S13", "DETACH DELETE: modo família, remover filmes de terror  (equivalente ao DELETE)")
TERROR = "MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'Terror'}) RETURN f.titulo ORDER BY f.titulo"
LIGACOES = "MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'Terror'}) MATCH (f)-[r]-() RETURN count(r)"
KUBRICK = """MATCH (:Pessoa {id: 'kubrick'})<-[:DIRIGIDO_POR]-(f:Filme)
RETURN f.titulo ORDER BY f.titulo"""

terror = [l[0] for l in executar(TERROR, tx)[1]]
print(f"  antes:  {len(terror)} filmes de terror — {', '.join(terror)}; "
      f"{valor(LIGACOES, tx)} relacionamentos ligados a eles")
print(f"          Kubrick dirigiu: {', '.join(l[0] for l in executar(KUBRICK, tx)[1])}")

tx.run("""
MATCH (f:Filme)-[:DO_GENERO]->(:Genero {id: 'Terror'})
DETACH DELETE f
""").consume()

print(f"  depois: {len(executar(TERROR, tx)[1])} filmes de terror; "
      f"{valor(LIGACOES, tx)} relacionamentos ligados a eles")
print(f"          Kubrick dirigiu: {', '.join(l[0] for l in executar(KUBRICK, tx)[1])}")

In [ ]:
tx.rollback()
sessao.close()

print("Rollback feito. Conferindo a base fora da transação:")
print("  Tenet existe?           ", valor("MATCH (f:Filme {id: 'tenet'}) RETURN count(f) > 0"))
print("  Filmes de terror:       ", valor("MATCH (:Filme)-[:DO_GENERO]->(g:Genero {id: 'Terror'}) RETURN count(*)"))
print("  Total de nós:           ", valor("MATCH (n) RETURN count(n)"), "(esperado 75)")
print("  Total de relacionamentos:", valor("MATCH ()-[r]->() RETURN count(r)"), "(esperado 149)")

## 6. Resumo

| Consulta | Equivale no MP3 a | Recurso Cypher | Travessia |
|---|---|---|---|
| T1 a T7 | `g.triples()` (os 7 padrões) | `MATCH` de um relacionamento | — |
| I1 | subClassOf (reasoner) | caminho `SUBGENERO_DE*` | sim |
| I2 | domain/range (reasoner) | papel pelo relacionamento, `UNION` | — |
| I3 | inverseOf (reasoner) | mesma aresta nos dois sentidos | — |
| I4 | TransitiveProperty (reasoner) | caminho `INFLUENCIOU*` | sim |
| I5 | SymmetricProperty (reasoner) | relacionamento sem direção | — |
| I6 | subPropertyOf + transitiva (reasoner) | caminho `SEQUENCIA_DE*` | sim |
| V1 | disjointWith (inconsistência) | verificação com `DISJUNTO_DE` | sim |
| V2 | AsymmetricProperty (inconsistência) | verificação de ciclo | sim |
| S1 | SELECT + FILTER | `WHERE` | sim |
| S2 | ORDER BY + LIMIT | `ORDER BY ... LIMIT` | — |
| S3 | agregação + HAVING | `count`/`avg` + `WITH ... WHERE` | — |
| S4 | agregação por gênero | `count`/`avg` | sim |
| S5 | VALUES + GROUP_CONCAT | `CASE` + `collect` | — |
| S6 | OPTIONAL | `OPTIONAL MATCH` + propriedade de aresta | — |
| S7 | caminho `influenciou+` | caminho variável `*` | sim |
| S8 | ASK antes × depois | `count(*) > 0`, um salto × caminho | sim |
| S9 | ASK com inversas | `count(*) > 0` | sim |
| S10 | CONSTRUCT para schema.org | mapa JSON-LD | — |
| S11 | INSERT DATA | `CREATE` | — |
| S12 | DELETE/INSERT | `SET` | — |
| S13 | DELETE | `DETACH DELETE` | — |

As atualizações rodam numa transação desfeita no fim, e a base no Memgraph continua igual à carga.

In [ ]:
driver.close()